In [ ]:
%load_ext autoreload
%autoreload 2

# ipyfernel

>  Connecting a solveit kernel to a Modal sandbox

In [ ]:
#| default_exp ipyfernel

In [ ]:
from solveit_modal.modal import *

In [ ]:
app = mk_app()
img = mk_img()
sb = mk_sandbox(app, img, gpu='T4')
host,port = get_tunnel(sb)
inject_pubkey(sb, get_pubkey()); start_ssh(sb)
ssh = mk_ssh(host, port)
verify_ssh(ssh)

⌛ creating sandbox…


✔ sandbox ready


✔ tunnel: r437.modal.host:40137


✔ public key injected


✔ started ssh service


System: Linux
Hostname: modal
User: root
Kernel: 4.4.0
Architecture: x86_64
OS Type: GNU/Linux
GPU: Tesla T4


In [ ]:
#| export
from time import sleep

In [ ]:
#| export
def start_remote_kernel(ssh,                            # SSH caller from mk_ssh
    kernel_path:str='/root/.local/share/jupyter/runtime/kernel-ipyf.json',  # kernel JSON path
    ) -> None:
    "Start remote kernel and wait until ready."
    ssh('mkdir', '-p', '/root/.local/share/jupyter/runtime')
    ssh(f'nohup python3 -m ipykernel_launcher -f {kernel_path} >/tmp/kernel.log 2>&1 &')
    for i in range(10):
        if 'ok' in ssh(f'test -f {kernel_path} && echo ok || echo no'): break
        sleep(0.2)
    log = ssh('cat /tmp/kernel.log')
    assert 'To connect another client' in log, f'kernel failed to start:\n{log}'
    print(f'✔ remote kernel ready: {kernel_path}')

In [ ]:
start_remote_kernel(ssh)

✔ remote kernel ready: /root/.local/share/jupyter/runtime/kernel-ipyf.json


In [ ]:
#| export
def get_remote_python(ssh,                       # SSH caller from mk_ssh
    ) -> str:                                    # path to remote python
    "Return the path of Python on the remote machine."
    return ssh('which python3')

In [ ]:
p = get_remote_python(ssh); p

'/usr/local/bin/python3'

In [ ]:
#| export
from ipyfernel.core import *

In [ ]:
register_remote_kernel(remote_python=p)
connect_existing_kernel(f'{host}:{port}')

ipyf_remote_kernel is already a registered kernel
/app/data/.ssh/config file updated.


Successfully created connection file and forwarded ports!


Success: connected to remote kernel via r437.modal.host:40137


In [ ]:
%%remote
import platform
platform.uname()

uname_result(system='Linux', node='modal', release='4.4.0', version='#1 SMP Sun Jan 10 15:06:54 PST 2016', machine='x86_64')


In [ ]:
%%remote
!nvidia-smi

Sat Jun  6 08:48:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       On  |   00000000:F4:00.0 Off |                    0 |
| N/A   30C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
stop_remote()

Code cells will now run locally.
Shutting down remote kernel


In [ ]:
sb.terminate()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()